# Person 2 — BM25, MRR and nDCG

This notebook uses the same Python modules and evaluator as the CLI. It starts with a **synthetic offline demo**. Change `USE_REAL_FINDER` below to evaluate the pinned FinDER snapshot. No LLM or API key is needed.


In [ ]:
from pathlib import Path
import os

if not Path("finder_bm25").is_dir() and Path("person2_bm25/finder_bm25").is_dir():
    os.chdir("person2_bm25")
assert Path("finder_bm25").is_dir(), "Run this notebook from the person2_bm25 folder"

from finder_bm25.bm25 import BM25Retriever
from finder_bm25.data import prepare_records, read_jsonl, save_bundle
from finder_bm25.download import fetch_records
from finder_bm25.evaluation import collect_run, evaluate_rag
from finder_bm25.metrics import mrr_at_k, ndcg_at_k
from finder_bm25.reporting import save_evaluation, table


## 1. Confirm the ranking metrics

Ranks 1, 5 and missed should produce MRR@5 = 0.4. Retrieving only one of two gold passages must give nDCG below 1.


In [ ]:
rankings = {"q1": ["gold"], "q2": ["a", "b", "c", "d", "gold"], "q3": []}
qrels = {qid: {"gold": 1} for qid in rankings}
print("MRR@5:", mrr_at_k(rankings, qrels, 5))
print("nDCG@5, one of two gold passages:", ndcg_at_k(["a"], {"a": 1, "b": 1}, 5))


## 2. Prepare the corpus and relevance judgments

Real data requires `python -m pip install -r requirements.txt` in the notebook kernel environment. All references are pooled before selecting evaluation queries. Default whole references give exact evidence-ID relevance labels.


In [ ]:
USE_REAL_FINDER = False
if USE_REAL_FINDER:
    records, source = fetch_records(Path("data/raw"))
else:
    records = read_jsonl(Path("examples/demo.jsonl"))
    source = {"kind": "synthetic_demo"}

bundle = prepare_records(records, seed=42, source=source)
save_bundle(bundle, Path("data/notebook_real" if USE_REAL_FINDER else "data/notebook_demo"))
print(bundle["manifest"])


## 3. Build BM25 and inspect evidence

The index sees reference text only. It does not receive gold answers, query labels or the gold company as a retrieval filter.


In [ ]:
import time
start = time.perf_counter()
retriever = BM25Retriever(bundle["corpus"], k1=1.2, b=0.75)
build_ms = (time.perf_counter() - start) * 1000
question = bundle["queries"][0]["text"]
documents = {doc["doc_id"]: doc for doc in bundle["corpus"]}
print("Question:", question)
for rank, hit in enumerate(retriever.search(question, top_k=5), 1):
    print(rank, round(hit["score"], 4), documents[hit["doc_id"]]["text"][:400])


## 4. Run the common evaluation at k = 3, 5, 10

Answer metrics remain N/A until Person 3 supplies generated answers and the shared correctness scorer. Latency measures search only.


In [ ]:
summaries = []
output = Path("results/notebook_real" if USE_REAL_FINDER else "results/notebook_demo")
for k in (3, 5, 10):
    run = collect_run(bundle, retriever, method="bm25", top_k=k,
                      partition="test" if USE_REAL_FINDER else "all",
                      config={"k1": 1.2, "b": 0.75}, build_latency_ms=build_ms)
    evaluation = evaluate_rag(bundle, run)
    save_evaluation(bundle, run, evaluation, output / f"k{k}")
    summaries.append(evaluation["summary"])
print(table(summaries))


## 5. Team handoff and interpretation

- Share the prepared bundle so Dense, BM25, Hybrid and MMR use identical text, IDs and qrels.
- Read `report.md` and `per_query.csv` for missed evidence and its first relevant rank.
- Use the CLI `compare` command when Person 1 provides dense rankings.
- Tune BM25 on the local dev partition, then evaluate the frozen configuration on test.
- These are **pooled reference-passage** results, not full 10-K retrieval.
- See `METHODOLOGY.md`, `ANALYSIS.md` and `RESULTS.md` for the full protocol and measured results.
